In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from birddog.tracker import (
    PageTracker,
    DynamoDBPageChangeLogTable,
    DynamoDBPageTrackerTable,
    SQLitePageChangeLogTable,
    SQLitePageTrackerTable,
    PageChangeLog,
    WikiDocTracker,
    )

from birddog.wiki import (
    get_recent_changes,
    lookup_namespace_id,
    get_recent_changes_v2,
    _api_url,
    )

from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import (
    DatabaseUpdater,
    )
from birddog.store import KeyValueStore, DynamoDBKeyValueStore
from birddog.utility import fetch_url, json_size, now, HeartbeatManager
#from birddog.log import get_logger

2026-02-10 09:01:25,045 [INFO] Translation is enabled. Using GCP translator
2026-02-10 09:01:25,046 [INFO] Using Google Cloud translation API
2026-02-10 09:01:25,046 [INFO] GoogleCloudTranslator using REST API
2026-02-10 09:01:25,057 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-02-10 09:01:25,340 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com


In [3]:
#def copy_page_tracker_to_ddb(batch_size=100, limit=None):
#    ddb_table = DynamoDBPageTrackerTable()
#    page_tracker = PageTracker()
#    entries = list(page_tracker._page_dict.items())
#    if not limit:
#        limit = len(entries)
#    print(f"pushing {limit} entries to DDB, batch_size={batch_size}")
#    for i in range(0, limit, batch_size):
#        print(f"batch {i}")
#        batch = { title: update for title, update in entries[i:(i+batch_size)] }
#        ddb_table.put(batch)

In [ ]:
#copy_page_tracker_to_ddb(batch_size=1000)

In [ ]:
#tracker = PageTracker()

In [ ]:
#changes = PageChangeLog()

In [ ]:
#wikisource_file_ns = "Файл"
#commons_file_ns = "File"
#commons_base = "https://commons.wikimedia.org"

In [ ]:
#lookup_namespace_id(wikisource_file_ns)

In [ ]:
#lookup_namespace_id(commons_file_ns)

In [ ]:
#c=get_recent_changes(cutoff_date="2026,02,03,23:00", base=commons_base, namespace=6)

In [ ]:
#len(c)

In [ ]:
#runtime = Runtime()

In [ ]:
#updater = DatabaseUpdater(runtime)

In [ ]:
from datetime import datetime, timedelta, timezone
import time

def _parse_utc(ts):
    if ts is None:
        return None
    if isinstance(ts, datetime):
        dt = ts
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    if isinstance(ts, str):
        s = ts.strip()
        if s.endswith("Z"):
            s = s[:-1] + "+00:00"
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    raise TypeError(f"Unsupported timestamp type: {type(ts)}")

def _format_utc_z(dt):
    dt = _parse_utc(dt)
    return dt.replace(microsecond=0).isoformat().replace("+00:00", "Z")

def _utc_now_dt():
    return datetime.now(timezone.utc)

def offset_utc(ts, seconds):
    dt = _parse_utc(ts)
    return _format_utc_z(dt + timedelta(seconds=seconds))
   

In [6]:
_WIKIMEDIA_COMMONS = "https://commons.wikimedia.org"
_UK_WIKISOURCE = "https://uk.wikisource.org"
_UK_WIKISOURCE_NS = "Файл"

In [ ]:
_WIKI_DOC_TRACKER_KV_TABLE = "doc_tracker"
_WIKI_DOC_TRACKER_HEARTBEAT_INTERVAL = 15  # seconds
_WIKI_SENTINEL = "WIKI_SENTINEL"
_WIKI_CHANGE_EVENT_WINDOW = 300  # seconds (default; can override via ctor)
_DOC_TABLE_SENTINEL = "DOC_SENTINEL"

class WikiDocTracker(HeartbeatManager):
    def __init__(
        self,
        cutoff_time=None,
        base_url=_WIKIMEDIA_COMMONS,
        namespace="File",
        db=None,
        change_window_s=_WIKI_CHANGE_EVENT_WINDOW,
    ):
        self._base_url = base_url
        self._namespace = namespace
        self._db = db if db else Database()

        self._namespace_id = lookup_namespace_id(self._namespace)
        self._kv = KeyValueStore(table_name=_WIKI_DOC_TRACKER_KV_TABLE)

        self._doc_kv_namespace = f"{self._base_url}:{self._namespace}"
        self._sentinel_kv_namespace = f"{self._base_url}:{self._namespace}:SENTINELS"

        self._cutoff_time = _format_utc_z(cutoff_time) if cutoff_time else None
        self._change_window_s = int(change_window_s)

        # In-memory doc index (normalized titles). None means "not loaded yet".
        self._doc_map = None  # dict[str, str] normalized_title -> link

        print(
            f"WikiDocTracker: base={self._base_url}, "
            f"namespace={self._namespace} (id={self._namespace_id})"
        )

        super().__init__(interval=_WIKI_DOC_TRACKER_HEARTBEAT_INTERVAL)

    def _reset(self):
        self._kv.remove_all(self._doc_kv_namespace)
        self._kv.remove_all(self._sentinel_kv_namespace)
        self._doc_map = None

    def _normalize_title(self, title):
        return title.replace(" ", "_")

    def _ensure_doc_map(self):
        if self._doc_map is None:
            self._doc_map = {}
            for k, v in self._kv.get_all(self._doc_kv_namespace):  # iterable of (key, value)
                self._doc_map[k] = v
    
            print(f"WikiDocTracker: loaded {len(self._doc_map)} doc titles into memory")

    def _store_relevant_titles(self, records):
        """
        Store relevant titles in KV and update in-memory doc_map incrementally.
        """
        inserts = 0
        for rec in records:
            link = rec.get("link", "")
            if not link.startswith(self._base_url):
                continue

            title = rec.get("title")
            if not title:
                continue

            nt = self._normalize_title(title)

            if nt in self._doc_map:
                continue

            self._kv.insert(self._doc_kv_namespace, nt, link)
            self._doc_map[nt] = link
            inserts += 1

        if inserts:
            print(f"WikiDocTracker: inserted {inserts} new doc titles")

    def _refresh_doc_titles(self):
        self._ensure_doc_map()
        try:
            doc_sentinel = self._kv.get(self._sentinel_kv_namespace, _DOC_TABLE_SENTINEL)
        except KeyError:
            doc_sentinel = None

        # Sort by descending CreatedAt (newest first).
        sort_spec = ("CreatedAt", False)

        newest_creation_date = None
        cursor = None
        while True:
            batch, cursor = self._db.scan("Documents", cursor=cursor, sort=sort_spec)
            if not batch:
                break

            if newest_creation_date is None:
                newest_creation_date = max(rec["CreatedAt"] for rec in batch if rec.get("CreatedAt"))

            self._store_relevant_titles(batch)

            # Stop when we reach records older than previously-seen newest doc.
            if doc_sentinel:
                last_created = batch[-1].get("CreatedAt")
                if last_created and last_created < doc_sentinel:
                    break

            if not cursor:
                break

        if newest_creation_date:
            self._kv.insert(self._sentinel_kv_namespace, _DOC_TABLE_SENTINEL, str(newest_creation_date))

    def _get_wiki_sentinel(self):
        """
        Returns UTC Z string.
        """
        try:
            t = self._kv.get(self._sentinel_kv_namespace, _WIKI_SENTINEL)
            t = _format_utc_z(t)
            if self._cutoff_time:
                t = _format_utc_z(max(_parse_utc(t), _parse_utc(self._cutoff_time)))
            return t
        except KeyError:
            if self._cutoff_time:
                return self._cutoff_time
            raise ValueError("undefined wiki sentinel")

    def _set_wiki_sentinel(self, timestamp):
        self._kv.insert(self._sentinel_kv_namespace, _WIKI_SENTINEL, _format_utc_z(timestamp))

    def _get_wiki_changes(self, utc_start):
        self._ensure_doc_map()

        start_dt = _parse_utc(utc_start)
        end_dt = min(start_dt + timedelta(seconds=self._change_window_s), _utc_now_dt())
        utc_start_z = _format_utc_z(start_dt)
        utc_end_z = _format_utc_z(end_dt)

        changes = get_recent_changes_v2(
            base=self._base_url,
            namespace=self._namespace_id,
            utc_start=utc_start_z,
            utc_end=utc_end_z,
        )

        if changes:
            print(f"found {len(changes)} changes")
            newest_seen = max(v["timestamp"] for v in changes.values())
            hits = { title for title in changes if title in self._doc_map }
            return hits, newest_seen
        return None, utc_end_z

    def _update_doc_records(self, doc_updates):
        """
        doc_updates: list[{"title": <normalized_title>, "link": <link>, "timestamp": <utc>, "user": <user>}]
        Placeholder for the DB update step.
        """
        # TODO: implement:
        # - resolve link/title -> document record id
        # - refresh metadata for those docs
        # - write updates to DB
        pass

    def heartbeat(self):
        # Incremental doc discovery (slow-changing)
        self._refresh_doc_titles()

        # Scan wiki changes window and collect hits
        last_sentinel = self._get_wiki_sentinel()
        print(
            f"WikiDocTracker: heartbeat start: wiki sentinel={last_sentinel}, "
            f"docs={len(self._doc_map) if self._doc_map is not None else 0}"
        )
        hits, next_sentinel = self._get_wiki_changes(last_sentinel)

        # Process hits -> update doc records
        if hits:
            doc_updates = []
            for nt, info in hits.items():
                link = self._doc_map.get(nt)
                if link:
                    doc_updates.append({ 
                        "title": nt, 
                        "link": link, 
                        "timestamp": info.get("timestamp"), 
                        "user":info.get("user") 
                    })
                    print(f"doc changed: {nt} timestamp={info['timestamp']} user={info.get('user')}")

            if doc_updates:
                self._update_doc_records(doc_updates)

        # Advance wiki sentinel (monotonic)
        if next_sentinel:
            self._set_wiki_sentinel(next_sentinel)
            print(f"WikiDocTracker: scanned window to {next_sentinel}, hits={len(hits) if hits else 0}")
        else:
            print(f"WikiDocTracker: no changes found.")
        print("WikiDocTracker: heartbeat finish")

In [4]:
runtime = Runtime()

2026-02-10 09:01:34,067 [INFO] PageUpdateManager.init(): detect_environment==local
2026-02-10 09:01:34,536 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2026-02-10 09:01:34,915 [INFO] KillSwitch: loading thresholds from resources/kill_thresholds.json
2026-02-10 09:01:34,918 [INFO] Runtime: truncating log history before 2026-01-26 17:01:34.918087+00:00


In [8]:
wdt = WikiDocTracker(
    runtime,
    base_url=_UK_WIKISOURCE,
    namespace=_UK_WIKISOURCE_NS,
)

2026-02-10 09:02:52,666 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2026-02-10 09:02:52,669 [INFO] WikiDocTracker: base=https://uk.wikisource.org, namespace=Файл (id=6)


In [ ]:
#wdt._reset()

In [9]:
wdt.heartbeat()

2026-02-10 09:03:00,450 [INFO] WikiDocTracker: loaded 4331 doc titles into memory
2026-02-10 09:03:02,720 [INFO] fetch_url: 4 requests in last 60s → 0.07 req/s
2026-02-10 09:03:03,829 [INFO] WikiDocTracker: heartbeat start: wiki sentinel=2026-02-09T16:33:12Z, docs=4331
2026-02-10 09:03:04,004 [INFO] WikiDocTracker: scanned window to 2026-02-09T16:38:12Z, hits=0
2026-02-10 09:03:04,006 [INFO] WikiDocTracker: heartbeat finish


In [ ]:
wdt.heartbeat()

In [ ]:
wdt.heartbeat()

In [ ]:
wdt.heartbeat()

In [ ]:
wdt.heartbeat()

In [ ]:
wdt.heartbeat()

In [ ]:
set('a')

In [ ]:
db = Database()

In [ ]:
dids = db.get_all_ids("Documents")

In [ ]:
doc_recs = db.read("Documents", dids)

In [ ]:
commons_prefix = "https://commons.wikimedia.org/wiki/File:"

In [ ]:
[d["title"] for d in doc_recs[:10] if d.get("link", "").startswith(commons_prefix)]

In [ ]:
commons_titles = { d["title"]: {"Id": d.get("Id"), "timestamp": d.get("timestamp")} 
                   for d in doc_recs 
                   if d.get("link", "").startswith(commons_prefix)
                 }

In [ ]:
#commons_titles

In [ ]:
commons_changes = [
    change for change in c 
    if change[0] in commons_titles
    ]

In [ ]:
commons_changes

In [ ]:
doc_store = KeyValueStore(table_name="doc_titles")
doc_ns = "commons"

In [ ]:
list(commons_titles.items())[:1]

In [ ]:
len(commons_titles)

In [ ]:
def store_titles(store, titles, ns=doc_ns):
    for title, entry in titles.items():
        store.insert(ns, title, str(entry.get("Id", "")))

In [ ]:
store_titles(doc_store, commons_titles)

In [ ]:
t = doc_store.get_all(doc_ns)